🎮 Ejercicio 1: "Sistema de Trading Asistido por IA"
Objetivo: Crear un sistema que analiza criptomonedas/acciones y recomienda operaciones.

Componentes:

Agente Investigador: Busca noticias, análisis técnico, sentimiento del mercado

Agente Analista Técnico: Usa múltiples modelos (DeepSeek, Gemini, GPT) para análisis

Agente de Riesgo: Con guardrails que bloquean operaciones peligrosas (>10% del portafolio, activos no verificados)

Agente Trader: Con handoff final que ejecuta (simula) la operación

Herramientas: WebSearch, calculadora de riesgo, simulador de operaciones

Salidas estructuradas: TradingSignal con confianza, stop-loss, take-profit

Reto extra: Implementar guardrails de salida que validen que las recomendaciones cumplan con límites de riesgo.

# 1-) Crear Base de Datos Vectoriales

In [1]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import os

C:\Users\Marco\OneDrive\Escritorio\Tirando.Codigo\Diplomado-IA\Practicas\Agentes\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# cargar datastore
loader = PyPDFLoader('./datastore/manual-trading-de-criptomonedas.pdf')

data = loader.load()

In [3]:
# dividir en chunks
splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", " ", ""],
    chunk_size=800,
    chunk_overlap=80
)

docs = splitter.split_documents(data)

In [4]:
# crear embeddings
embedding_function = HuggingFaceEmbeddings(
    model_name="sentence-transformers/distiluse-base-multilingual-cased-v2",
    model_kwargs={"device" : "cuda"}
)

In [5]:
# base de datos vectorial
vectorstore = Chroma.from_documents(
    docs,
    embedding=embedding_function,
    persist_directory="chroma_db/"
)

print(f'Base de datos creada con {len(docs)} chunks')
print('Guardada en: chroma_db/')

Base de datos creada con 153 chunks
Guardada en: chroma_db/


## 1.1-) Expandir DB

In [7]:
PERSIST_DIR = "chroma_db"

PDF_PATHS = [
    "./datastore/uso-de-criptomonedas-como-alternativa-financiera.pdf",
    "./datastore/analisis-riesgos-implicitos-criptoactivos.pdf",
    "./datastore/Evaluacion_del_rendimiento_de_inversiones_en_criptos.pdf",
    "./datastore/analisis-riesgo-rendimiento-criptomonedas.pdf"
]

In [8]:
# 1) Cargar el vectorstore existente
try:
    vectorstore = Chroma(
        persist_directory=PERSIST_DIR,
        embedding_function=embedding_function
    )

    print(f'✅ Base de Datos cargada correctamente: {PERSIST_DIR}')

except Exception as ex:
    print(f'⚠️ Error al cargar la Base de Datos: {PERSIST_DIR}, Exception message -> {str(ex)}')

✅ Base de Datos cargada correctamente: chroma_db


In [9]:
# 2) Cargar nuevos PDF
for pdf_path in PDF_PATHS:
    loader = PyPDFLoader(pdf_path)
    data = loader.load()
    docs = splitter.split_documents(data)

    # Agregar metadata para identificar la fuente
    for d in docs:
        d.metadata = d.metadata or {}
        d.metadata["source_pdf"] = os.path.basename(pdf_path)

    # Agregar al vectorstore
    vectorstore.add_documents(docs)
    print(f"✅ Agregados {len(docs)} chunks desde: {pdf_path}")

print(f"📁 BD actualizada en: {PERSIST_DIR}/")

✅ Agregados 433 chunks desde: ./datastore/uso-de-criptomonedas-como-alternativa-financiera.pdf
✅ Agregados 72 chunks desde: ./datastore/analisis-riesgos-implicitos-criptoactivos.pdf
✅ Agregados 37 chunks desde: ./datastore/Evaluacion_del_rendimiento_de_inversiones_en_criptos.pdf
✅ Agregados 459 chunks desde: ./datastore/analisis-riesgo-rendimiento-criptomonedas.pdf
📁 BD actualizada en: chroma_db/


# 2-) Configurar Cliente

In [10]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.prompts import ChatPromptTemplate
from langgraph.prebuilt import create_react_agent

In [11]:
load_dotenv(override=True)
openai_api_key = os.getenv("OPENAI_API_KEY")

if openai_api_key:
    print(f'OPENAI_API_KEY: {openai_api_key[:8]}')
else:
    print('OPENAI_API_KEY not found')

OPENAI_API_KEY: sk-proj-


# *Agente DataBase Managger* (RAG)

In [12]:
@tool
def buscar_en_bd(pregunta: str) -> str:
    """Busca información en la base de datos vectorial sobre gestión de cryptomonedas, evaluación de rendimiento de inversiones en criptos y análisis de riesgos."""
    docs = vectorstore.similarity_search(pregunta, k=5)
    return "\n\n".join([doc.page_content for doc in docs])

In [14]:
#crear el prompt de systema
system_prompt_db_managger = ChatPromptTemplate.from_messages([
    ("system", """Eres un agente de IA experto en Trading de Cripto-Activos, tu deber es usar la herramienta 'buscar_en_bd' cuando sea necesario\

### Eres el asistente virtual de una empresa llamada Condorsito:

1. Actúa como un asesor financiero senior con 15 años de experiencia en trading de criptomonedas
2. El usuario te puede hacer preguntas sobre riesgos de inversiones en cryptos, evaluacion de rendimiento o sobre el uso de criptomonedas y tu deber es buscar en la base de datos usando la herramienta 'buscar_en_bd'\
3. USA la herramienta "buscar_en_bd" para buscar informacion en la base de datos vectorial.
4. Responde basándote en lo que encontraste
5. Sé cálido, empático y usa emojis apropiados
6. Ofrece ejemplos prácticos
7. Indica de que libro obtuviste la respuesta, hazlo como referencia y citalo en formato apa 7ma edicion.
8. Si el usuario se desvia del tema o pregunta sobre otros temas que no sea sobre finanzas cripto solo responde `Lo siento, solo se sobre finanzas cripto`
9. Tu unico tema de conversacion y/o ayuda sera sobre finanzas cripto y nadamas
10. Debes evitar tecnicismos si el usuario tiene poco o nulo conocimiento sobre finanzas y/o cripto trading
11. Evitar confundir al usuario y siempre preguntar que tanto nivel de conocimiento tiene sobre el tema que preguntó
12. Ofrece ayuda adicional sobre tecnicismos si el usuario lo pide

Si no encuentras información en los libros, sé honesto y ofrece alternativas."""),
    ("placeholder", "{messages}")
])

In [15]:
#crear el agente
db_managger_llm = ChatOpenAI(
    model="gpt-4o-mini",
    api_key=openai_api_key,
    temperature=0.7
)

db_managger_agent = create_react_agent(
    db_managger_llm,
    [buscar_en_bd],
    prompt=system_prompt_db_managger
)

C:\Users\Marco\AppData\Local\Temp\ipykernel_22508\4226609284.py:8: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  db_managger_agent = create_react_agent(


# *Agente Predictivo* (Modelos de Machine Learning)

In [ ]:
import joblib
import pandas as pd

In [ ]:
@tool
def load_model(model_name, models_dir="models"):
    """ Carga un pipeline entrenado desde el directorio especificado."""
    model_files = {
        "Ethereum": "pipeline_Ethereum.pkl",
        "Bitcoin": "pipeline_Bitcoin.pkl"
    }

    if model_name.capitalize() not in model_files.keys():
        print(f"⚠️load_model:  Modelo '{model_name.capitalize()}' no está definido.")
        return None

    path = os.path.join(models_dir, model_files[model_name.capitalize()])

    if os.path.exists(path):
        return joblib.load(path)
    else:
        print(f"⚠️ ️load_model: No se encontró el archivo: {path}")
        return None

In [ ]:
@tool
def predict_crypto_currency(features: dict, pipeline: object) -> float:
    """ Predecir el precio de Cripto-Activos usando el pipeline entrenado. """
    try:
        if features and pipeline:
            X = pd.DataFrame([features])
            prediction = pipeline.predict(X)
            return float(prediction)
        else:
            raise Exception('Pipeline Model or New Features are Null')

    except Exception as ex:
        print('⚠️ predict_crypto_currency: Excepcion inesperada ->  ',str(ex))
        return float(0)

In [ ]:
system_prompt_predictions = ChatPromptTemplate.from_messages([
    ("system", """Eres un agente de IA experto en Trading de Cripto-Activos, tu deber es usar la herramienta 'load_model' para cargar dinámicamente el modelo de Machine Learning correspondiente a la moneda que el usuario indique, y después usar la herramienta 'predict_crypto_currency' para realizar predicciones sobre las nuevas entradas que el usuario proporcione.

### Eres el asistente virtual de una empresa llamada Condorsito:

1. Actúa como un asesor financiero senior con 15 años de experiencia en trading de criptomonedas
2. Tu deber es cargar el modelo adecuado con la herramienta 'load_model' según la moneda que el usuario especifique.
3. Una vez cargado el modelo, usa la herramienta 'predict_crypto_currency' para generar la predicción con las entradas proporcionadas por el usuario.
4. Si el usuario no especifica la moneda, siempre pregúntale cuál desea usar antes de ejecutar 'load_model'.
5. Es obligatorio el uso de herramientas: nunca respondas con predicciones sin haber ejecutado 'load_model' y 'predict_crypto_currency'.
6. Ofrece ejemplos prácticos de cómo interpretar las predicciones.
7. Debes evitar tecnicismos si el usuario tiene poco o nulo conocimiento sobre finanzas y/o cripto trading.
8. Evita confundir al usuario y siempre pregunta qué nivel de conocimiento tiene sobre el tema que consultó.
9. Si no encuentras el modelo con 'load_model' explica: 'no tengo acceso a ese modelo predictivo de Machine Learning'.

Si no encuentras información sobre el modelo o no puedes ejecutar la predicción, sé honesto y ofrece alternativas."""),
    ("placeholder", "{messages}")
])

In [ ]:
#crear el agente
prediction_llm = ChatOpenAI(
    model="gpt-4o-mini",
    api_key=openai_api_key,
    temperature=0.7
)

prediction_agent = create_react_agent(
    prediction_llm,
    [load_model, predict_crypto_currency],
    prompt=system_prompt_predictions
)

# *Agente Investigador* (Deep Research)

In [16]:
from langchain_community.utilities import GoogleSerperAPIWrapper

In [17]:
serper = GoogleSerperAPIWrapper()

In [18]:
serper.run("Cual es el precio actual de Bitcoin en pesos mexicanos")

'El precio actual de Bitcoin es de 1.174.354,74 MXN por BTC. Con un suministro circulante de 19.992.809 BTC, significa que Bitcoin tiene una capitalización ... BTC/MXN - Bitcoin Peso mexicano ; Compra/Venta: 1.173.596,00 ; Vol. (24h): 32,96 ; Cap. mercado: 1,37 ; Rango día: 1.013.937 ; 52 semanas: 964.396 ... 1 Bitcoin cuesta actualmente Mex$1.16M Mex$ MXN, 1.5% en las últimas 24 h. Calcula el cambio de Bitcoin (BTC) a Mexican Peso (MXN) con nuestro conversor de ... 1 BTC es igual a 1,159,916.67 MXN empleando el tipo de cambio medio actual del mercado de $1,159,916.67. Si quieres enviar 1 BTC a MXN, comprueba si Xe podría ... Precio en vivo y calculadora de Bitcoin (BTC). btc logo Bitcoin. 1,176,560 MXN. ↑ 1.6%20 feb 2026. ¿Qué valor tiene 1 Bitcoin en MXN? En este momento, el precio de 1 Bitcoin (BTC) en Mexican Peso (MXN) es aproximadamente de MX$1.173.233. ¿Cuántos BTC ... Convierte 1 Bitcoin (BTC) a peso mexicano (MXN) al instante con nuestro conversor de criptomonedas. Actualmente

In [ ]:
@tool
def search_tool(query: str) -> str:
    """
    Herramienta de investigación profunda para el agente de criptomonedas.
    - Usa Google Serper API para buscar noticias, análisis técnico y sentimiento del mercado.
    - Es obligatoria para obtener información externa (deep research).
    """
    try:
        result = serper.run(query)
        if not result:
            return "⚠️ search_tool: No se encontraron resultados relevantes en la búsqueda."
        return result
    except Exception as ex:
        return f"⚠️  search_tool: Error en la búsqueda con Serper: {str(ex)}"

In [ ]:
system_prompt_deep_research = ChatPromptTemplate.from_messages([
    ("system", """Eres un agente de IA experto en análisis técnico de criptomonedas.
Tu deber es usar la herramienta 'search_tool' para investigar noticias, análisis técnico y sentimiento del mercado cuando el usuario lo requiera.

### Eres el asistente virtual de una empresa llamada Condorsito:

1. Actúa como un investigador senior con 15 años de experiencia en trading y análisis de criptomonedas.
2. El usuario puede pedirte información sobre noticias recientes, análisis técnico o sentimiento del mercado.
3. No puedes gestionar una búsqueda por tu cuenta: SIEMPRE ES OBLIGATORIO usar la herramienta 'search_tool' para obtener información externa.
4. Usa la herramienta 'search_tool' **solo cuando sea necesario** y únicamente si no se encuentra información en la base de datos interna del RAG.
5. Responde basándote en lo que encontraste con la herramienta, nunca inventes información.
6. Sé cálido, empático y usa emojis apropiados.
7. Ofrece ejemplos prácticos de cómo interpretar las noticias o el sentimiento del mercado en relación con criptomonedas.
8. Indica de qué fuente obtuviste la información, cítala en formato APA 7ma edición.
9. Si el usuario se desvía del tema o pregunta sobre otros temas que no sean finanzas cripto, responde únicamente: `Lo siento, solo sé sobre finanzas cripto`.
10. Tu único tema de conversación y ayuda será sobre finanzas cripto y nada más.
11. Evita tecnicismos si el usuario tiene poco o nulo conocimiento sobre finanzas y/o cripto trading.
12. Siempre pregunta qué nivel de conocimiento tiene el usuario sobre el tema que consultó, para adaptar tu explicación.
13. Ofrece ayuda adicional sobre tecnicismos si el usuario lo pide.

Si no encuentras información en la base de datos ni en la web, sé honesto y ofrece alternativas."""),
    ("placeholder", "{messages}")
])

In [ ]:
search_llm = ChatOpenAI(
    model="gpt-4o-mini",
    api_key=openai_api_key,
    temperature=0.7
)

search_agent = create_react_agent(
    search_llm,
    [search_tool],
    prompt=system_prompt_deep_research
)

# *Agente de Riesgo* (Guardrails)

In [ ]:
from pydantic import BaseModel, Field, ValidationError

In [ ]:
# Formato Estructurado
class TradingSignal(BaseModel):
    confianza: float = Field(description="Nivel de confianza de la operación de trading (0.0 a 1.0)", ge=0.0, le=1.0)
    stop_loss: float = Field(description="Precio límite para detener pérdidas en la operación", gt=0.0)
    take_profit: float = Field(description="Precio objetivo para asegurar ganancias en la operación", gt=0.0)

In [ ]:
# Lista de activos verificados
CRYPTO_ACTIVOS_DISPONIBLES = {"BTC", "ETH"}

@tool
def output_guardrail_trading_signal(signal: TradingSignal, asset: str, porcentaje_portafolio: float) -> TradingSignal:
    """
    Guardrail para validar señales de trading:
    - Bloquea operaciones que superen el 10% del portafolio.
    - Bloquea activos no verificados.
    - Valida estructura con Pydantic.
    """
    try:
        # Validación de porcentaje de portafolio
        if porcentaje_portafolio > 0.10:
            raise ValueError("⚠️ Output Guardrail: operación rechazada porque supera el 10% del portafolio.")

        # Validación de activo verificado
        if asset not in CRYPTO_ACTIVOS_DISPONIBLES:
            raise ValueError(f"⚠️ Output Guardrail: operación rechazada porque el activo '{asset}' no está disponible.")

        # Validación de estructura con Pydantic
        validated_signal = TradingSignal(**signal.dict())

        return validated_signal

    except ValidationError as ve:
        raise ValueError(f"⚠️ Output Guardrail: error de validación en TradingSignal → {ve}")
    except Exception as e:
        raise ValueError(f'⚠️ Output Guardrail: error inesperado -> {str(e)}')

In [ ]:
system_prompt_risk_agent = ChatPromptTemplate.from_messages([
    ("system", """Eres un agente de IA especializado en **gestión de riesgo en trading de criptomonedas**.
Tu deber es usar la herramienta `output_guardrail_trading_signal` para validar cualquier señal de trading antes de entregarla al usuario.

### Contexto:
- Trabajas como asistente virtual de una empresa llamada Condorsito.
- Tu rol es proteger al usuario de operaciones riesgosas o inválidas.
- El formato de salida siempre debe seguir la clase `TradingSignal`:
  - confianza: float (0.0 a 1.0)
  - stop_loss: float (>0.0)
  - take_profit: float (>0.0)

### Reglas del agente de Riesgo:
1. **Siempre** valida las señales con el guardrail `output_guardrail_trading_signal`.
2. Bloquea operaciones que superen el 10% del portafolio.
3. Bloquea activos no verificados en la lista de `CRYPTO_ACTIVOS_DISPONIBLES`.
4. Si la señal no pasa las validaciones, responde con un mensaje claro: `"Guardrail: operación rechazada porque..."`.
5. Nunca inventes datos de señales: si no tienes información suficiente, dilo explícitamente.
6. Sé claro, directo y evita tecnicismos innecesarios, salvo que el usuario pida más detalle.
7. Tu único tema de conversación es **gestión de riesgo en trading de criptomonedas**.
8. Si el usuario pregunta sobre otro tema, responde únicamente: `Lo siento, solo sé sobre gestión de riesgo en trading de criptomonedas`.
9. Pregunta siempre el nivel de conocimiento del usuario para adaptar tu explicación.
10. Ofrece ejemplos prácticos de cómo aplicar stop-loss y take-profit en escenarios de riesgo.

### Objetivo:
Tu misión es garantizar que todas las señales de trading que entregues estén verificadas y seguras, usando el guardrail `output_guardrail_trading_signal` como filtro obligatorio."""),
    ("placeholder", "{messages}")
])

In [ ]:
risk_llm = ChatOpenAI(
    model="gpt-4o-mini",
    api_key=openai_api_key,
    temperature=0.7
)

risk_agent = create_react_agent(
    risk_llm,
    [output_guardrail_trading_signal],
    prompt=system_prompt_risk_agent,
)


# *Agente de Reporte* (Pushover/Sendgrid)

In [ ]:
import sendgrid
from sendgrid.helpers.mail import Mail, Email, To, Content
from typing import Dict
import requests

In [ ]:
@tool
def send_html_email(subject: str, html_body: str) -> Dict[str, str]:
    """ Envía un correo electrónico con el asunto y el cuerpo HTML al usuario, registrando sus operaciones realizadas con criptomonedas."""
    sg = sendgrid.SendGridAPIClient(api_key=os.getenv('SENDGRID_API_KEY'))
    from_email = Email("marco.salram.04@gmail.com")  # Cambiar a tu remitente verificado
    to_email = To("#######")  # Cambiar a su receptor
    content = Content("text/html", html_body)
    mail = Mail(from_email, to_email, subject, content).get()
    response = sg.client.mail.send.post(request_body=mail)
    return {"status": "success"}

In [ ]:
token_user = os.getenv("PUSHOVER_USER")
token_pass = os.getenv("PUSHOVER_TOKEN")
pushover_url = 'https://api.pushover.net/1/messages.json'

print(f'PUSHOVER_USER: {token_user[:8]}')
print(f'PUSHOVER_TOKEN: {token_pass[:8]}')

In [ ]:
@tool
def push(message):
    """ Envía notificación al dispositivo móvil del usuario para avisarle el envío de su reporte de criptomonedas a su correo electrónico. """
    print(f'Push Message: {message}')
    payload = {"user" : token_user, "token" : token_pass, "message" : message}
    requests.post(pushover_url, data=payload)

In [ ]:
html_istructions = ChatPromptTemplate.from_messages([
    ("system", """Tu deber es convertir un cuerpo de correo electronico de texto a un cuerpo de correo electronico HTML. \
    Se te proporciona un cuerpo de correo electronico de texto que puede tener algun markdown y necesitas convertirlo a un cuerpo\
         de correo electronico HTML con un diseño claro, atractivo y que se adecuado para el asunto."""),
     ("placeholder", "{messages}")
])

html_converter_llm = ChatOpenAI(
    model="gpt-4o-mini",
    api_key=openai_api_key,
    temperature=0.7
)

html_converter_agent = create_react_agent(
    html_converter_llm,
    prompt=html_istructions
)

In [ ]:
@tool
def html_converter_tool(email_body: str) -> str:
     """ Convierte un cuerpo de correo electrónico en texto/markdown a HTML semántico y atractivo. """
     result =  html_converter_agent.invoke({"input": email_body})
     return result["output"]

In [ ]:
subject_writer_instructions = ChatPromptTemplate.from_messages([
    ("system", """Tu deber es escribir un asunto para un correo electrónico transaccional.
Se te proporciona un mensaje y necesitas redactar un asunto claro, profesional y confiable
para un correo que contiene un reporte de operaciones en criptomonedas."""),
    ("placeholder", "{messages}")
])

subject_writer_llm = ChatOpenAI(
    model="gpt-4o-mini",
    api_key=openai_api_key,
    temperature=0.7
)

subject_writer_agent = create_react_agent(
    subject_writer_llm,
    prompt=subject_writer_instructions
)

In [ ]:
@tool
def subject_writer_tool(subject: str) -> str:
    """ Escribe un asunto para un correo electrónico transaccional que contiene un reporte de operaciones en criptomonedas"""
    result = subject_writer_agent.invoke({"input": subject})
    return result["output"]

In [ ]:
system_prompt_report = ChatPromptTemplate.from_messages([
    ("system", """Eres un formateador y remitente de correos electronicos.\
    Recibes el cuerpo de un correo electronico para enviarlo.\
    Primero usas la herrameinta 'subject_writer_tool' para escribir un asunto para el correo electronico (reporte de operaciones del usuario con criptomonedas),\
    luego usas la herramienta 'html_converter_tool' para convertir el cuerpo a HTML.\
    Finalmente, usas la herramienta 'send_html_email' para enviar el correo electronico con el asunto y el cuerpo HTML y usas la herramienta 'push' para mandar\
    una notificación al dispositivo del usuario para avisarle que le llegó un reporte de sus operaciones con criptomonedas a su correo electronico"""),
    ("placeholder", "{messages}")
])

report_tools = [subject_writer_tool, html_converter_tool, send_html_email, push]

report_llm = ChatOpenAI(
    model="gpt-4o-mini",
    api_key=openai_api_key,
    temperature=0.7
)

report_agent = create_react_agent(
    report_llm,
    tools=report_tools,
    prompt=system_prompt_report
)

### Tips: Mejorar Docstrings y reducir el system prompt
### Agente Evaluador para comparar informacion del RAG / Deep Search